# 01 - Data audit
First look at the raw NTSB data before cleaning: which years it covers, what the codes mean, and what needs fixing.


In [2]:
from pathlib import Path
import warnings

import pandas as pd
import pyodbc

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
pd.set_option("display.max_columns", None)

In [3]:
db_path = Path("../data/raw/avall.mdb").resolve()
conn = pyodbc.connect(r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};Dbq=" + str(db_path) + ";")

pd.read_sql("SELECT TOP 5 * FROM events", conn)

,ev_id,ntsb_no,ev_type,ev_date,ev_dow,ev_time,ev_tmzn,ev_city,ev_state,ev_country,ev_site_zipcode,ev_year,ev_month,mid_air,on_ground_collision,latitude,longitude,latlong_acq,apt_name,ev_nr_apt_id,ev_nr_apt_loc,apt_dist,apt_dir,apt_elev,wx_brief_comp,wx_src_iic,wx_obs_time,wx_obs_dir,wx_obs_fac_id,wx_obs_elev,wx_obs_dist,wx_obs_tmzn,light_cond,sky_cond_nonceil,sky_nonceil_ht,sky_ceil_ht,sky_cond_ceil,vis_rvr,vis_rvv,vis_sm,wx_temp,wx_dew_pt,wind_dir_deg,wind_dir_ind,wind_vel_kts,wind_vel_ind,gust_ind,gust_kts,altimeter,wx_dens_alt,wx_int_precip,metar,ev_highest_injury,inj_f_grnd,inj_m_grnd,inj_s_grnd,inj_tot_f,inj_tot_m,inj_tot_n,inj_tot_s,inj_tot_t,invest_agy,ntsb_docket,ntsb_notf_from,ntsb_notf_date,ntsb_notf_tm,fiche_number,lchg_date,lchg_userid,wx_cond_basic,faa_dist_office,dec_latitude,dec_longitude
0,20080211X00175,DFW08RA039,ACC,2008-01-10,Th,1907,UTC,Pitt Meadows,NaN,CA,NaN,2008,1,NaN,NaN,049130N,0122412W,None,NaN,NaN,NaN,0.0,None,NaN,None,NaN,NaN,0,NaN,NaN,0,NaN,NaN,NaN,0,0,NaN,None,None,NaN,0,0,0,U,NaN,F,F,0,0.000000,None,None,None,NONE,NaN,NaN,NaN,0,0,2,0,2,O,None,None,None,None,None,2025-09-23 18:50:18,dobn,NaN,None,49.216634,-122.683870
1,20080107X00026,SEA08LA057,ACC,2008-01-01,Tu,2230,UTC,Sonoma,CA,USA,95476,2008,1,Y,N,381326N,1222659W,None,Schellville/Sonoma,OQ3,ONAP,0.0,None,10.0,None,WFAC,2254.0,220,DVO,2.0,7,UTC,DAYL,CLER,0,0,NONE,None,None,10.0,57,32,80,U,10.0,F,T,0,30.190001,None,None,None,NONE,NaN,NaN,NaN,0,0,3,0,3,N,None,None,None,None,None,2020-09-25 18:05:31,NaN,VMC,None,38.223888,-122.449722
2,20080109X00036,DFW08CA054,ACC,2008-01-01,Tu,2200,UTC,Arcola,TX,USA,77583,2008,1,NaN,NaN,293022N,0952836W,None,HOUSTON-SOUTHWEST,AXH,ONAP,0.0,None,68.0,None,WFAC,1553.0,0,SGR,NaN,0,NaN,DAYL,CLER,0,0,NONE,None,None,10.0,55,14,360,U,15.0,F,F,20,30.620001,None,None,None,NONE,NaN,NaN,NaN,0,0,1,0,1,N,None,None,None,None,None,2020-09-25 18:05:31,NaN,VMC,None,29.506111,-95.476669
3,20080107X00027,DFW08LA055,ACC,2008-01-03,Th,825,UTC,Oklahoma City,OK,USA,73008,2008,1,NaN,NaN,035321N,0973829W,None,Wiley Post,PWA,ONAP,0.0,None,1299.0,None,WFAC,853.0,0,PWA,1299.0,0,UTC,NITE,CLER,0,0,NONE,None,None,10.0,0,0,170,U,16.0,F,T,0,30.639999,None,None,None,FATL,1.0,0.0,0.0,0,0,8,0,9,NaN,None,None,None,None,None,2020-09-25 18:04:02,NaN,VMC,None,35.533611,-97.641387
4,20080115X00051,DEN08CA047,ACC,2008-01-02,We,2230,UTC,Loveland,CO,USA,80538,2008,1,NaN,NaN,040276N,0010504W,None,Fort Collins Loveland,FNL,ONAP,1.0,None,5016.0,None,WFAC,2235.0,180,KFNL,NaN,1,UTC,DAYL,CLER,0,0,NONE,None,None,10.0,19,7,150,U,5.0,F,T,0,30.260000,None,None,None,NONE,NaN,NaN,NaN,0,0,1,0,1,N,None,None,None,None,None,2020-09-25 18:06:16,NaN,VMC,None,40.451667,-105.001113


## 1. How many events, and from which years?

In [4]:
pd.read_sql("""
    SELECT COUNT(*)     AS n_events,
           MIN(ev_year) AS first_year,
           MAX(ev_year) AS last_year
    FROM events
""", conn)

,n_events,first_year,last_year
0,31124,2008,2026


In [5]:
pd.read_sql("""
    SELECT ev_year, COUNT(*) AS n_events
    FROM events
    GROUP BY ev_year
    ORDER BY ev_year
""", conn)

,ev_year,n_events
0,2008,1892
1,2009,1783
2,2010,1786
3,2011,1848
4,2012,1834
5,2013,1561
6,2014,1534
7,2015,1581
8,2016,1662
9,2017,1635
